[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/numerical_methods/08_numerical_ode_solvers/exercises.ipynb)

# Exercises — Topic 08: Numerical ODE Solvers

20 fully solved problems in 4 levels: concept checks, foundational computations and derivations, AI/ML and physics applications, and challenge proofs.

## Level 0 — Concept Check

### Problem L0.1: Local versus global truncation error

A method has local truncation error $O(h^{3})$ per step. What is its global error over a fixed interval $[0, T]$, and why? What order is the method called?

**Solution.**

**The counting argument.** Covering $[0, T]$ with step $h$ takes $N = T/h$ steps. If each step introduces a fresh defect of size $Ch^{3}$ and those defects merely accumulated, the total would be

$$
N \cdot Ch^{3} = \frac{T}{h}\cdot Ch^{3} = CTh^{2} = O(h^{2}) .
$$

**One order is always lost** in passing from per-step defect to accumulated error, because the number of steps grows like $1/h$.

**Why "merely accumulate" is not automatic.** Each earlier error is then *propagated* by the dynamics, being multiplied by roughly $(1 + hL)$ per subsequent step, where $L$ is the Lipschitz constant of $f$. The discrete Gronwall lemma shows this replaces the plain sum by

$$
\Vert e_N \Vert \le \frac{Ch^{2}}{L}\left(e^{LT} - 1\right),
$$

still $O(h^{2})$, but with a constant that grows exponentially in $LT$. Stability is exactly the hypothesis that keeps this factor finite.

**Naming.** The convention names a method by its *global* order, and the defect $O(h^{p+1})$ per step corresponds to $\tau = O(h^{p})$ after dividing by $h$. A per-step defect $O(h^{3})$ is therefore an order-$2$ method — e.g. Heun, the midpoint rule, or the trapezoidal rule.

$$
\boxed{\text{local defect } O(h^{p+1}) \Longrightarrow \text{global error } O(h^{p}); \quad O(h^{3}) \text{ per step} \Rightarrow \text{order } 2}
$$

*Key takeaway:* Always ask which error is being quoted; the useful number is the global one, and it is always one power of $h$ weaker than the per-step defect.

### Problem L0.2: Why can no explicit Runge–Kutta method be A-stable?

Explain, using the structure of the stability function, why A-stability is unattainable for explicit Runge–Kutta methods and attainable for implicit ones.

**Solution.**

**Explicit case.** In an explicit $s$-stage RK method each stage $k_i$ is a finite combination of previously computed stages. Applying it to $y' = \lambda y$ and writing $z = h\lambda$, each $k_i$ becomes a *polynomial* in $z$ times $y_n$, so

$$
y_{n+1} = R(z)y_n, \qquad R(z) = 1 + z + \frac{z^2}{2!} + \cdots + \frac{z^{p}}{p!} + (\text{higher terms}),
$$

a **polynomial of degree at most $s$**. A nonconstant polynomial satisfies $\vert R(z)\vert \to \infty$ as $\vert z\vert \to \infty$ in every direction, including along the negative real axis. Hence $\{ z : \vert R(z)\vert \le 1\}$ is a *bounded* set and cannot contain the unbounded left half-plane: **no explicit RK method is A-stable.** Its stability region is always a bounded blob near the origin, e.g. $(-2, 0)$ on the real axis for Euler and Heun, $(-2.7853, 0)$ for RK4.

**Implicit case.** Here $\mathbf{y}_{n+1}$ appears inside the stage equations, so solving the linear test problem requires *inverting* a matrix depending on $z$, and $R(z)$ comes out as a **rational** function $P(z)/Q(z)$. Rational functions can stay bounded at infinity: backward Euler has $R(z) = (1-z)^{-1} \to 0$, and the trapezoidal rule has $R(z) = \frac{1+z/2}{1-z/2} \to -1$. Both satisfy $\vert R\vert \le 1$ on the whole left half-plane.

$$
\boxed{\text{explicit} \Rightarrow R \text{ polynomial} \Rightarrow \text{bounded stability region} \Rightarrow \text{never A-stable}}
$$

*Key takeaway:* The explicit/implicit distinction is not a matter of taste or convenience — it is the algebraic difference between polynomial and rational stability functions, and it is what forces stiff problems onto implicit solvers.

### Problem L0.3: What makes a problem stiff?

Which of these are stiff on $[0, 10]$? (a) $y' = -y$; (b) $\mathbf{y}' = \begin{bmatrix} -1 & 0 \\ 0 & -1000\end{bmatrix}\mathbf{y}$; (c) $y' = 1000y$; (d) $y'' + y = 0$ written as a first-order system. Justify with the spectrum, and state the step restriction forward Euler would face in each case.

**Solution.**

Stiffness compares the step size demanded by *stability* with the one demanded by *accuracy*.

**(a) Not stiff.** Single eigenvalue $\lambda = -1$. Forward Euler needs $h \lt 2$; accuracy on $[0,10]$ needs $h \approx 0.1$ anyway. Stability is not binding.

**(b) Stiff.** Eigenvalues $-1$ and $-1000$, stiffness ratio $1000$. The fast mode $e^{-1000t}$ has vanished by $t = 0.01$ and contributes nothing thereafter, but forward Euler must obey $h \lt 2/1000 = 0.002$ *for the whole interval* or the numerical fast component explodes: $5000$ steps where accuracy would justify about $100$.

**(c) Not stiff — unstable.** $\lambda = +1000 \gt 0$: the true solution grows like $e^{1000t}$. No method is "stable" here and none should be; the small step is genuinely required for accuracy. Stiffness is a property of *decaying* modes only. This distinction matters: a tiny step for a genuinely fast solution is not waste.

**(d) Not stiff — oscillatory.** As a system, $\begin{bmatrix} 0 & 1 \\ -1 & 0\end{bmatrix}$ has eigenvalues $\pm i$, purely imaginary. Forward Euler's stability region touches the imaginary axis only at the origin, so it is *unconditionally unstable* here (the amplitude grows by $\sqrt{1+h^2}$ per step regardless of $h$) — a different pathology from stiffness, cured by a symplectic or trapezoidal method rather than by an implicit stiff solver.

$$
\boxed{\text{stiff} \iff \text{decaying modes force } h \ll h_{\text{accuracy}}; \quad \text{(b) only, ratio } 1000, \ h \lt 0.002}
$$

*Key takeaway:* Stiffness is about *fast decaying* transients constraining the step long after they have died; fast growth and pure oscillation are entirely different problems requiring entirely different cures.

### Problem L0.4: Choosing a solver

Pick a method for each: (a) a smooth nonstiff two-body orbit integrated for 10 periods to 6 digits; (b) the same system integrated for $10^{8}$ periods to study long-term stability; (c) a 60-species combustion mechanism with rate constants spanning $10^{10}$; (d) a model with a discontinuous forcing term switching at $t = 3$.

**Solution.**

**(a) Adaptive explicit RK — Dormand–Prince (`RK45`/`ode45`).** Nonstiff and smooth, so an explicit method is cheapest; the embedded pair gives per-step error control at six evaluations per step, automatically shrinking $h$ near perihelion and stretching it at apoapsis.

**(b) A symplectic integrator — Störmer–Verlet or a higher-order Yoshida splitting.** Over $10^{8}$ periods the issue is no longer per-step accuracy but *secular drift*. RK4's energy error grows linearly with $t$, manufacturing a fake orbital decay; a symplectic method conserves a modified Hamiltonian $\tilde{H} = H + O(h^{p})$ exactly, so the energy error stays bounded forever. A low-order symplectic method beats a high-order non-symplectic one here.

**(c) An implicit stiff solver — BDF (`ode15s`, SciPy `BDF`) or Radau IIA.** Stiffness ratio $\sim 10^{10}$ makes any explicit method require $\sim 10^{10}$ steps. BDF is stable to order 6 and its Newton solves exploit the sparse Jacobian of the reaction network.

**(d) Event detection with a restart.** Stepping *across* a discontinuity destroys the order — every method silently degrades to first order or worse, because its Taylor expansion assumes smoothness. The correct handling is to detect the switching time with a root find (Topic 02), integrate exactly up to $t = 3$, and restart the integration with the new right-hand side. `solve_ivp` provides this through its `events` argument.

$$
\boxed{\text{(a) RK45 adaptive, (b) symplectic Verlet, (c) BDF/Radau, (d) event detection + restart}}
$$

*Key takeaway:* Order is only one of four axes; the others are stability (stiffness), structure preservation (Hamiltonian), and smoothness (events) — and each of them can matter more than order.

## Level 1 — Foundation

### Problem L1.1: Forward Euler by hand

Apply forward Euler with $h = 0.2$ to $y' = y - t^{2} + 1$, $y(0) = 0.5$, for three steps. The exact solution is $y(t) = (t+1)^{2} - \tfrac12 e^{t}$. Tabulate the errors and confirm they grow roughly linearly.

**Solution.**

The scheme is $y_{n+1} = y_n + h\,(y_n - t_n^2 + 1)$.

**Step 1** ($t_0 = 0$, $y_0 = 0.5$): $f = 0.5 - 0 + 1 = 1.5$, so $y_1 = 0.5 + 0.2(1.5) = 0.8$.

**Step 2** ($t_1 = 0.2$, $y_1 = 0.8$): $f = 0.8 - 0.04 + 1 = 1.76$, so $y_2 = 0.8 + 0.2(1.76) = 1.152$.

**Step 3** ($t_2 = 0.4$, $y_2 = 1.152$): $f = 1.152 - 0.16 + 1 = 1.992$, so $y_3 = 1.152 + 0.2(1.992) = 1.5504$.

| $t_n$ | $y_n$ (Euler) | $y(t_n)$ exact | error $y_n - y(t_n)$ |
| :--- | :--- | :--- | :--- |
| $0.2$ | $0.800000$ | $0.829299$ | $-0.029299$ |
| $0.4$ | $1.152000$ | $1.214088$ | $-0.062088$ |
| $0.6$ | $1.550400$ | $1.648941$ | $-0.098541$ |

**Reading the errors.** They grow by roughly $0.033$, $0.036$ per step — steadily, and slightly faster than linearly because this problem has $f_y = 1 \gt 0$, so earlier errors are amplified by $(1 + h) = 1.2$ each step in addition to the fresh local defect. The first local defect matches theory: $\tfrac{h^2}{2}y''(0) = \tfrac{0.04}{2}(1.5) = 0.03$ against the observed $0.0293$, since $y'' = y' - 2t$ gives $y''(0) = 1.5$.

$$
\boxed{y_1 = 0.8, \quad y_2 = 1.152, \quad y_3 = 1.5504; \quad \text{errors } -0.0293,\ -0.0621,\ -0.0985}
$$

*Key takeaway:* Euler always lags a convex solution (it follows the tangent, which lies below the curve), and the error accumulates with amplification $(1 + hf_y)$ per step — the mechanism the Gronwall bound formalizes.

### Problem L1.2: Backward Euler and the trapezoidal rule on a decaying problem

Integrate $y' = -2y$, $y(0) = 1$, to $t = 0.5$ with $h = 0.1$ using forward Euler, backward Euler, and the trapezoidal rule. Compare with the exact $e^{-1}$ and explain the sign pattern of the errors.

**Solution.**

Each method has a closed-form amplification factor on this linear problem, with $z = h\lambda = -0.2$:

$$
\text{FE: } R = 1 + z = 0.8, \qquad \text{BE: } R = \frac{1}{1-z} = \frac{1}{1.2}, \qquad \text{TR: } R = \frac{1 + z/2}{1 - z/2} = \frac{0.9}{1.1} .
$$

After $5$ steps, $y_5 = R^{5}$:

| Method | $R$ | $y_5 = R^{5}$ | error vs $e^{-1} = 0.367879$ |
| :--- | :--- | :--- | :--- |
| Forward Euler | $0.800000$ | $0.327680$ | $-0.040199$ |
| Backward Euler | $0.833333$ | $0.401878$ | $+0.033998$ |
| Trapezoidal | $0.818182$ | $0.366648$ | $-0.001231$ |
| Exact | $e^{-0.2} = 0.818731$ | $0.367879$ | — |

**Why the signs.** Compare each $R$ with $e^{z} = e^{-0.2} = 0.818731$:

- $1 + z = 0.8 \lt e^{z}$ — Euler over-damps and **undershoots**, since $e^{z} = 1 + z + \tfrac{z^2}{2} + \cdots$ and the missing $\tfrac{z^2}{2} \gt 0$.
- $(1-z)^{-1} = 1 + z + z^2 + \cdots = 0.8333 \gt e^{z}$ — backward Euler under-damps and **overshoots** (its $z^2$ coefficient is $1$ rather than $\tfrac12$).
- $\frac{1+z/2}{1-z/2} = 1 + z + \tfrac{z^2}{2} + \tfrac{z^3}{4} + \cdots$ matches $e^{z}$ through $z^{2}$, so it is **second order**, and its error is $\left(\tfrac14 - \tfrac16\right)z^{3} = \tfrac{z^3}{12} \lt 0$, a small undershoot.

The trapezoidal error is $33\times$ smaller than Euler's, consistent with $O(h^2)$ against $O(h)$ at $h = 0.1$.

$$
\boxed{\text{FE } 0.327680,\quad \text{BE } 0.401878,\quad \text{TR } 0.366648,\quad \text{exact } 0.367879}
$$

*Key takeaway:* On a linear problem every one-step method is just a rational approximation $R(z)$ to $e^{z}$; the order is how many Taylor coefficients of $e^{z}$ it reproduces, and the stability is where $\vert R \vert \le 1$.

### Problem L1.3: One step of classical RK4

Take one RK4 step of size $h = 0.2$ for $y' = y - t^{2} + 1$, $y(0) = 0.5$. Report the four stage slopes and compare the result with Euler and Heun on the same step.

**Solution.**

With $f(t,y) = y - t^2 + 1$, $t_0 = 0$, $y_0 = 0.5$, $h = 0.2$:

$$
\begin{aligned}
k_1 &= f(0,\ 0.5) = 0.5 - 0 + 1 = 1.5, \\
k_2 &= f(0.1,\ 0.5 + 0.1 \cdot 1.5) = f(0.1,\ 0.65) = 0.65 - 0.01 + 1 = 1.64, \\
k_3 &= f(0.1,\ 0.5 + 0.1 \cdot 1.64) = f(0.1,\ 0.664) = 0.664 - 0.01 + 1 = 1.654, \\
k_4 &= f(0.2,\ 0.5 + 0.2 \cdot 1.654) = f(0.2,\ 0.8308) = 0.8308 - 0.04 + 1 = 1.7908 .
\end{aligned}
$$

$$
y_1 = y_0 + \frac{h}{6}\left(k_1 + 2k_2 + 2k_3 + k_4\right) = 0.5 + \frac{0.2}{6}\left(1.5 + 3.28 + 3.308 + 1.7908\right) = 0.5 + \frac{0.2}{6}(9.8788) = 0.8292933 .
$$

Comparison against the exact $y(0.2) = 1.44 - \tfrac12 e^{0.2} = 0.8292986$:

| Method | Evaluations | $y_1$ | error | order of local defect |
| :--- | :--- | :--- | :--- | :--- |
| Forward Euler | $1$ | $0.800000$ | $-2.93 \times 10^{-2}$ | $O(h^{2})$ |
| Heun (RK2) | $2$ | $0.826000$ | $-3.30 \times 10^{-3}$ | $O(h^{3})$ |
| Classical RK4 | $4$ | $0.8292933$ | $-5.29 \times 10^{-6}$ | $O(h^{5})$ |

Four times the work buys a $5500\times$ smaller error on this step. Note also that the weights $(\tfrac16, \tfrac13, \tfrac13, \tfrac16)$ are exactly Simpson's rule: when $f$ does not depend on $y$, RK4 *is* Simpson's rule applied to $\int f\,dt$.

$$
\boxed{k = (1.5,\ 1.64,\ 1.654,\ 1.7908), \qquad y_1 = 0.8292933, \qquad \text{error} = -5.29\times 10^{-6}}
$$

*Key takeaway:* RK4 samples the slope at the start, twice at the midpoint, and at the end, then weights them like Simpson's rule — high order from repeated evaluations of $f$ inside a single step, with no derivatives of $f$ required.

### Problem L1.4: Step-size limits from absolute stability

For $y' = -100y$ determine the largest step forward Euler, Heun, and RK4 may take, and contrast with backward Euler. How many steps does each need on $[0, 1]$?

**Solution.**

Stability requires $z = h\lambda$ to lie in the method's region, here on the negative real axis with $\lambda = -100$.

**Forward Euler.** $\vert 1 + z\vert \le 1 \iff -2 \le z \le 0$, so

$$
h \le \frac{2}{100} = 0.02 \implies N \ge 50 \text{ steps on } [0,1],
$$

each costing 1 evaluation: $50$ evaluations.

**Heun / RK2.** $R(z) = 1 + z + \tfrac{z^2}{2}$; solving $\vert R(z)\vert = 1$ on the negative axis gives $z = -2$ exactly (since $1 + z + z^2/2 = -1 \Rightarrow z^2 + 2z + 4 = 0$ has no real root, while $=+1$ gives $z(1 + z/2)=0$, so $z=-2$). Same limit $h \le 0.02$, but at $2$ evaluations per step: $100$ evaluations — *worse* than Euler per unit work when stability, not accuracy, is binding.

**RK4.** $R(z) = 1 + z + \tfrac{z^2}{2} + \tfrac{z^3}{6} + \tfrac{z^4}{24}$; the real stability boundary is $z \approx -2.7853$, so

$$
h \le \frac{2.7853}{100} = 0.027853 \implies N \ge 36 \text{ steps}, \ 144 \text{ evaluations} .
$$

**Backward Euler.** $\vert 1 - z\vert \ge 1$ holds for **every** $h \gt 0$ when $\lambda \lt 0$. Stability imposes no restriction at all; $h$ is chosen purely for accuracy — say $h = 0.1$, i.e. $10$ steps, each requiring one linear (or Newton) solve.

$$
\boxed{\text{FE, Heun: } h \le 0.02; \quad \text{RK4: } h \le 0.027853; \quad \text{BE: any } h \gt 0}
$$

*Key takeaway:* Higher order buys only a marginally larger stability region ($2 \to 2.785$) while costing four times the evaluations — order does not solve stiffness, and only implicit methods do.

### Problem L1.5: Deriving the two-stage Runge–Kutta order conditions

Find all values of $(\alpha, b_1, b_2)$ for which the method $k_1 = f(t_n, y_n)$, $k_2 = f(t_n + \alpha h,\ y_n + \alpha h k_1)$, $y_{n+1} = y_n + h(b_1k_1 + b_2k_2)$ is second order, and name three members of the family.

**Solution.**

**Exact expansion.** From $y' = f$ and the chain rule $y'' = f_t + f_y f$:

$$
y(t_n + h) = y_n + hf + \frac{h^{2}}{2}\left(f_t + f_y f\right) + O(h^{3}), \qquad \text{all terms at } (t_n, y_n).
$$

**Method expansion.** Taylor-expand $k_2$ in two variables about $(t_n, y_n)$:

$$
k_2 = f + \alpha h f_t + (\alpha h f) f_y + O(h^{2}) = f + \alpha h\left(f_t + f_y f\right) + O(h^2),
$$

so

$$
y_{n+1} = y_n + h(b_1 + b_2)f + h^{2}\,b_2\alpha\left(f_t + f_y f\right) + O(h^{3}) .
$$

**Matching.** The $O(h)$ terms give $b_1 + b_2 = 1$ (consistency), and the $O(h^{2})$ terms give $b_2\alpha = \tfrac12$. Two equations, three unknowns — a **one-parameter family** of second-order methods:

$$
b_2 = \frac{1}{2\alpha}, \qquad b_1 = 1 - \frac{1}{2\alpha}, \qquad \alpha \neq 0 .
$$

| Name | $\alpha$ | $b_1$ | $b_2$ | Form |
| :--- | :--- | :--- | :--- | :--- |
| Heun (explicit trapezoid) | $1$ | $\tfrac12$ | $\tfrac12$ | $y_n + \tfrac{h}{2}(k_1 + k_2)$ |
| Midpoint | $\tfrac12$ | $0$ | $1$ | $y_n + hk_2$ |
| Ralston | $\tfrac23$ | $\tfrac14$ | $\tfrac34$ | minimizes the order-3 error bound |

**Why order 3 is unreachable.** At $O(h^{3})$ the exact solution contains the independent elementary differential $f_y(f_t + f_yf)$, which two stages cannot generate at all — the resulting condition would be $0 = \tfrac16$. Getting order 3 needs three stages, and order 5 needs six (not five) — the Butcher barriers.

$$
\boxed{b_1 + b_2 = 1, \quad b_2\alpha = \tfrac12: \text{ Heun } (\alpha=1), \text{ midpoint } (\alpha=\tfrac12), \text{ Ralston } (\alpha=\tfrac23)}
$$

*Key takeaway:* Order conditions come from matching Taylor coefficients term by term; the free parameters left over are what distinguishes members of a family, and they are usually spent minimizing the leading error constant or enlarging the stability region.

### Problem L1.6: Verifying the observed order of a method

Integrate $y' = -y$, $y(0) = 1$ to $t = 1$ (exact value $e^{-1} = 0.3678794$) with forward Euler, the trapezoidal rule, and RK4 at $h = 0.1$ and $h = 0.05$, and extract the observed orders.

**Solution.**

On this linear problem each method gives $y_N = R(z)^{N}$ with $z = -h$, $N = 1/h$, so the errors can be computed in closed form:

$$
\text{FE: } (1-h)^{1/h}, \qquad \text{TR: } \left(\frac{1 - h/2}{1 + h/2}\right)^{1/h}, \qquad \text{RK4: } \left(1 - h + \tfrac{h^2}{2} - \tfrac{h^3}{6} + \tfrac{h^4}{24}\right)^{1/h} .
$$

| Method | $E(0.1)$ | $E(0.05)$ | ratio $E(h)/E(h/2)$ | $p_{\text{obs}} = \log_2(\text{ratio})$ |
| :--- | :--- | :--- | :--- | :--- |
| Forward Euler | $1.9201\times 10^{-2}$ | $9.3935\times 10^{-3}$ | $2.044$ | $1.03$ |
| Trapezoidal | $3.0690\times 10^{-4}$ | $7.6662\times 10^{-5}$ | $4.003$ | $2.00$ |
| Classical RK4 | $3.3324\times 10^{-7}$ | $1.9976\times 10^{-8}$ | $16.68$ | $4.06$ |

The observed orders $1$, $2$, $4$ match the theory. The small excess above the nominal value (e.g. $1.03$ rather than $1.00$) is the higher-order term in the error expansion $E(h) = C_1h^{p} + C_2h^{p+1} + \cdots$, and it shrinks as $h$ does: refining once more to $h = 0.025$ gives $E = 4.6470\times 10^{-3}$ for Euler, ratio $2.021$, $p_{\text{obs}} = 1.015$.

**Why this test matters.** Halving $h$ and checking the error ratio is the standard verification of an ODE-solver implementation. A method that produces plausible-looking output but the *wrong* observed order almost always has a mistyped tableau coefficient — and this test finds it in seconds, whereas eyeballing a trajectory does not.

$$
\boxed{p_{\text{obs}} = \log_2\frac{E(h)}{E(h/2)}: \quad 1.03 \text{ (Euler)}, \quad 2.00 \text{ (trapezoid)}, \quad 4.06 \text{ (RK4)}}
$$

*Key takeaway:* Order is an empirically measurable property; always verify it before trusting a solver, and remember that measuring it requires errors well above the round-off floor and well below the asymptotic regime's breakdown.

## Level 2 — Applications in AI/ML & Physics

### Problem L2.1: The true cost of stiffness

The linearized model $\mathbf{y}' = A\mathbf{y}$ has eigenvalues $\lambda_1 = -1$ and $\lambda_2 = -1000$, integrated on $[0, 10]$ to $1\%$ accuracy. Compare forward Euler and backward Euler in steps and in real work.

**Solution.**

**What accuracy alone would demand.** After $t \approx 0.01$ the fast mode $e^{-1000t}$ is below $10^{-4}$ and irrelevant. The remaining solution is $\sim e^{-t}$, and $1\%$ accuracy from a first-order method needs a local relative error $\sim h/2 \approx 0.01$, i.e. $h \approx 0.02$ — or, being generous with a smooth slow mode, $h \approx 0.1$ giving $100$ steps.

**What stability actually demands of forward Euler.** The *stiff* eigenvalue governs the restriction:

$$
h \le \frac{2}{\vert\lambda_2\vert} = \frac{2}{1000} = 0.002 \implies N = \frac{10}{0.002} = 5000 \text{ steps} .
$$

If $h$ exceeds $0.002$ by even a little, the numerical fast component is multiplied by $\vert 1 + h\lambda_2\vert \gt 1$ every step and blows up — at $h = 0.003$, by $2^{3333} \approx 10^{1003}$ over the interval. **The step is $50$ times smaller than accuracy requires, purely to keep a component that is numerically zero from exploding.**

**Backward Euler.** A-stable, so $h$ is chosen by accuracy alone: $h = 0.1$, $N = 100$ steps. Each step requires solving $(I - hA)\mathbf{y}_{n+1} = \mathbf{y}_n$ — one $d \times d$ LU factorization, reusable across steps since $A$ and $h$ are constant here, and $O(d^2)$ per step thereafter.

| Method | $h$ | Steps | Work per step | Verdict |
| :--- | :--- | :--- | :--- | :--- |
| Forward Euler | $0.002$ | $5000$ | 1 evaluation | stability-limited, $50\times$ wasted |
| Backward Euler | $0.1$ | $100$ | 1 linear solve (factorization reused) | accuracy-limited |

The advantage grows without bound with the stiffness ratio: at ratio $10^{6}$ (routine in combustion) forward Euler would need $5\times 10^{6}$ steps and is simply not an option.

$$
\boxed{\text{FE: } h \le 0.002,\ 5000 \text{ steps}; \quad \text{BE: } h = 0.1,\ 100 \text{ steps}; \quad \text{ratio grows with stiffness}}
$$

*Key takeaway:* On a stiff problem the implicit solve per step is nearly free compared with the thousands of explicit steps it replaces — this is the entire economic argument for implicit methods.

### Problem L2.2: An adaptive step-size controller in action

A fourth-order solver with an embedded fifth-order estimate is at $t_n$ with $h = 0.1$, tolerance $\mathrm{tol} = 10^{-6}$, and safety factor $\theta = 0.9$. It measures $\mathrm{err} = 10^{-4}$. What happens? Then the retried step measures $\mathrm{err} = 10^{-9}$; what is the next step size?

**Solution.**

**The controller.** For a method of order $p$ the local error scales as $\mathrm{err} \propto h^{p+1}$, so demanding $\mathrm{tol}$ from a new step $h_{\text{new}}$ gives

$$
\frac{\mathrm{tol}}{\mathrm{err}} = \left(\frac{h_{\text{new}}}{h}\right)^{p+1} \implies h_{\text{new}} = \theta\, h \left(\frac{\mathrm{tol}}{\mathrm{err}}\right)^{1/(p+1)} , \qquad p + 1 = 5 .
$$

**First attempt.** $\mathrm{err} = 10^{-4} \gt \mathrm{tol} = 10^{-6}$, so the step is **rejected**: the solution is *not* advanced, and

$$
h_{\text{new}} = 0.9 \times 0.1 \times \left(\frac{10^{-6}}{10^{-4}}\right)^{1/5} = 0.09 \times (10^{-2})^{0.2} = 0.09 \times 10^{-0.4} = 0.09 \times 0.39811 = 0.03583 .
$$

**Second attempt** at $h = 0.03583$ gives $\mathrm{err} = 10^{-9} \lt \mathrm{tol}$: the step is **accepted**, $t$ advances, and the next step size is

$$
h_{\text{new}} = 0.9 \times 0.03583 \times \left(\frac{10^{-6}}{10^{-9}}\right)^{1/5} = 0.03225 \times 10^{0.6} = 0.03225 \times 3.98107 = 0.12839 .
$$

This lies within the usual growth clamp $h_{\text{new}} \le 5h = 0.1791$, so it is used as is.

**Why the safety factor and the clamps.** $\theta \approx 0.9$ makes the *next* step likely to be accepted, since rejections cost a full evaluation set with nothing to show. The clamps ($0.2 \le h_{\text{new}}/h \le 5$) prevent wild oscillation when the error estimate is noisy — for instance near a discontinuity, where the asymptotic model $\mathrm{err} \propto h^{p+1}$ is simply false. Real solvers additionally use a *mixed* tolerance $\mathrm{tol}_i = \mathrm{atol} + \mathrm{rtol}\vert y_i\vert$ so that components of very different magnitude are each treated sensibly, and normalize $\mathrm{err}$ as an RMS over components.

$$
\boxed{\text{reject, } h \to 0.0358; \quad \text{then accept, } h \to 0.1284}
$$

*Key takeaway:* Adaptive control is a feedback loop on the *local* error using the model $\mathrm{err} \propto h^{p+1}$; global accuracy is not controlled directly, it follows from stability plus a uniformly small local error.

### Problem L2.3: Gradient descent is forward Euler, and the learning rate is a stability limit

Show that gradient descent is forward Euler applied to gradient flow, and derive the maximum learning rate for a quadratic loss with Hessian eigenvalues in $[1, 100]$. Give the optimal rate and the resulting convergence factor.

**Solution.**

**The correspondence.** Gradient flow is the ODE

$$
\dot{\boldsymbol{\theta}}(t) = -\nabla L(\boldsymbol{\theta}(t)) ,
$$

whose trajectories descend the loss continuously. Applying forward Euler with step $h = \eta$ gives

$$
\boldsymbol{\theta}_{k+1} = \boldsymbol{\theta}_k - \eta\,\nabla L(\boldsymbol{\theta}_k) ,
$$

**exactly** gradient descent. The learning rate *is* the step size.

**Stability analysis.** For $L(\boldsymbol{\theta}) = \tfrac12\boldsymbol{\theta}^{\top}H\boldsymbol{\theta}$ with $H = H^{\top} \succ 0$, the flow is $\dot{\boldsymbol{\theta}} = -H\boldsymbol{\theta}$, a linear system with eigenvalues $\lambda_i = -\mu_i$ where $\mu_i \in [1, 100]$ are the eigenvalues of $H$. Diagonalizing decouples the problem into scalar test equations, and forward Euler's condition $\vert 1 + \eta\lambda_i\vert \lt 1$ becomes

$$
\vert 1 - \eta\mu_i \vert \lt 1 \iff 0 \lt \eta \lt \frac{2}{\mu_i} \quad \forall i \iff \eta \lt \frac{2}{\mu_{\max}} = \frac{2}{100} = 0.02 .
$$

This is precisely the textbook optimization bound $\eta \lt 2/L_{\text{smooth}}$ — **the learning-rate limit is an absolute-stability limit**, and the divergence seen at too large a learning rate is the same oscillating blow-up as an unstable Euler integration.

**Optimal rate and convergence factor.** The contraction factor is $\max_i \vert 1 - \eta\mu_i\vert$, minimized by equalizing the extremes, $1 - \eta\mu_{\min} = -(1 - \eta\mu_{\max})$:

$$
\eta^{*} = \frac{2}{\mu_{\min} + \mu_{\max}} = \frac{2}{101} \approx 0.0198, \qquad \rho = \frac{\mu_{\max} - \mu_{\min}}{\mu_{\max} + \mu_{\min}} = \frac{\kappa - 1}{\kappa + 1} = \frac{99}{101} \approx 0.9802 ,
$$

with $\kappa = 100$. Reaching a relative error of $10^{-6}$ therefore needs $\ln(10^{-6})/\ln(0.9802) \approx 691$ iterations.

**Consequences of the ODE view.** *Stiffness in the loss landscape* is exactly a large $\kappa$: the fast directions force a small $\eta$ while the slow directions determine how long training takes. Preconditioning (Adam, natural gradient, batch normalization) is the machine-learning analogue of an implicit method — it reduces the effective stiffness so a larger step becomes stable. Momentum corresponds to the second-order heavy-ball ODE $\ddot{\boldsymbol{\theta}} + a\dot{\boldsymbol{\theta}} + \nabla L = 0$, and Nesterov acceleration to a time-varying damping $a(t) = 3/t$.

$$
\boxed{\eta \lt 2/\mu_{\max} = 0.02; \quad \eta^{*} = 2/101 \approx 0.0198; \quad \rho = 99/101 \approx 0.9802 \Rightarrow 691 \text{ steps for } 10^{-6}}
$$

*Key takeaway:* Optimization convergence theory and ODE absolute-stability theory are the same theory; "the learning rate diverged" and "the explicit method was unstable" are the same sentence.

### Problem L2.4: Neural ODEs and the adjoint sensitivity method

State the Neural ODE forward model, derive the adjoint ODE governing $\mathbf{a}(t) = \partial L/\partial\mathbf{h}(t)$, and explain what the method buys and what it costs numerically.

**Solution.**

**Forward model.** A Neural ODE replaces a stack of residual blocks by a continuous flow

$$
\frac{d\mathbf{h}(t)}{dt} = f_\theta(\mathbf{h}(t), t), \qquad \mathbf{h}(0) = \mathbf{x}, \qquad \text{output } \mathbf{h}(T), \qquad L = \ell(\mathbf{h}(T)) .
$$

Depth becomes integration time; the number of "layers" is whatever the adaptive solver chooses.

**Adjoint derivation.** Define $\mathbf{a}(t) = \dfrac{\partial L}{\partial \mathbf{h}(t)}$, the sensitivity of the loss to a perturbation of the state at time $t$. Perturbing the state at time $t$ by $\delta$ and propagating one infinitesimal step gives $\mathbf{h}(t + \varepsilon) \mapsto \mathbf{h}(t+\varepsilon) + \left(I + \varepsilon\dfrac{\partial f_\theta}{\partial\mathbf{h}}\right)\delta + O(\varepsilon^2)$, so by the chain rule

$$
\mathbf{a}(t)^{\top} = \mathbf{a}(t + \varepsilon)^{\top}\left(I + \varepsilon\frac{\partial f_\theta}{\partial \mathbf{h}}\right) + O(\varepsilon^{2}) .
$$

Rearranging and letting $\varepsilon \to 0$:

$$
\frac{d\mathbf{a}(t)}{dt} = -\left(\frac{\partial f_\theta}{\partial\mathbf{h}}\right)^{\top}\mathbf{a}(t), \qquad \mathbf{a}(T) = \frac{\partial \ell}{\partial\mathbf{h}(T)} .
$$

This is a **backward-in-time linear ODE** — continuous-time backpropagation, with the transposed Jacobian playing the role of the reverse-mode VJP. The parameter gradient is then another quadrature, integrated backward alongside it:

$$
\frac{dL}{d\theta} = -\int_{T}^{0}\mathbf{a}(t)^{\top}\frac{\partial f_\theta}{\partial\theta}\,dt .
$$

In practice one integrates the augmented system $(\mathbf{h}, \mathbf{a}, \partial L/\partial\theta)$ backward from $t = T$ in a single solver call, and each right-hand-side evaluation is one VJP through $f_\theta$ — no more expensive than a standard backward pass.

**What it buys.** $O(1)$ memory in depth: activations are *not* stored, since $\mathbf{h}(t)$ is reconstructed by integrating backward. Compare with backpropagating through the solver's internal steps, which stores every stage of every step.

**What it costs, numerically.** (i) Reconstructing $\mathbf{h}$ backward is itself an ODE solve and can be unstable — errors in the reverse trajectory contaminate the gradient, and dissipative forward dynamics become *anti*-dissipative in reverse. (ii) Solver tolerance becomes gradient noise: too loose a tolerance yields biased gradients that silently degrade training. (iii) The backward system can be stiff even when the forward one is not. Practical remedies are checkpointing (store $\mathbf{h}$ at a few times and re-integrate between them), tighter backward tolerances than forward, and *reversible* integrators — for instance the leapfrog/symplectic schemes of Problem L2.6 — which reconstruct the forward trajectory exactly.

$$
\boxed{\frac{d\mathbf{a}}{dt} = -\left(\frac{\partial f_\theta}{\partial\mathbf{h}}\right)^{\top}\mathbf{a}, \quad \mathbf{a}(T) = \nabla_{\mathbf{h}(T)}\ell; \qquad \frac{dL}{d\theta} = -\int_T^0 \mathbf{a}^{\top}\frac{\partial f_\theta}{\partial\theta}\,dt}
$$

*Key takeaway:* The adjoint method is backpropagation written as an ODE, so every numerical concern of this topic — order, stability, stiffness, tolerance — becomes a concern about the *gradients*, not merely about the forward pass.

### Problem L2.5: Diffusion-model sampling as ODE integration

Explain why a diffusion sampler is an ODE solver, and compare the function-evaluation budget of a first-order sampler with a second-order (Heun-type) one for the same sample quality.

**Solution.**

**The probability-flow ODE.** A score-based generative model defines a forward noising SDE $d\mathbf{x} = f(\mathbf{x},t)\,dt + g(t)\,d\mathbf{w}$. Song et al. (2021) showed that the deterministic ODE

$$
\frac{d\mathbf{x}}{dt} = f(\mathbf{x},t) - \tfrac12 g(t)^{2}\,\nabla_{\mathbf{x}}\log p_t(\mathbf{x})
$$

has **the same time-marginal distributions** $p_t$ as the SDE. Sampling therefore means integrating this ODE backward from $t = T$ (pure noise) to $t = 0$ (data), with the learned score network $s_\theta(\mathbf{x},t) \approx \nabla_{\mathbf{x}}\log p_t(\mathbf{x})$ supplying the right-hand side. **The number of denoising steps is literally the number of solver steps, and each step's right-hand side is one network evaluation (NFE)** — which dominates the cost entirely.

**Order versus NFE.** For a target sample error $\epsilon$, an order-$p$ method needs $N \sim (C/\epsilon)^{1/p}$ steps. A first-order (Euler/DDIM-style) sampler uses $1$ NFE per step; a Heun-type second-order sampler uses $2$ NFE per step but needs far fewer steps:

| Sampler | Order | NFE per step | Steps for a given quality | Total NFE |
| :--- | :--- | :--- | :--- | :--- |
| Ancestral DDPM | $1$ (stochastic) | $1$ | $\sim 1000$ | $\sim 1000$ |
| DDIM (Euler on the flow ODE) | $1$ | $1$ | $\sim 50$ | $\sim 50$ |
| Heun / EDM second order | $2$ | $2$ | $\sim 18$ | $\sim 36$ |
| DPM-Solver (exponential, order 2–3) | $2$–$3$ | $1$–$2$ | $\sim 10$–$20$ | $\sim 10$–$20$ |

Second order wins decisively once the required accuracy is modest-to-tight, because halving the error costs $\sqrt{2}\times$ more steps rather than $2\times$. This is the standard order-versus-cost trade of Topic 08, applied where a single function evaluation costs a full U-Net forward pass.

**Two refinements that matter here.** (i) The ODE is *semilinear*, $\mathbf{x}' = a(t)\mathbf{x} + b(t)s_\theta(\mathbf{x},t)$: the linear part can be integrated **exactly** with an integrating factor, so only the nonlinear remainder needs a numerical rule. That is the exponential-integrator idea behind DPM-Solver, and it is why it beats generic RK at the same NFE. (ii) The time discretization is not uniform: EDM uses a warped schedule $\sigma_i$ concentrating steps where the score changes fastest — a hand-tuned analogue of adaptive step control, chosen offline because per-step error estimation would cost extra NFEs.

$$
\boxed{\text{sampling} = \text{integrating } \tfrac{d\mathbf{x}}{dt} = f - \tfrac12 g^{2}s_\theta; \quad \text{order 2 at } \sim 36 \text{ NFE beats order 1 at } \sim 50}
$$

*Key takeaway:* Nearly all "fast sampling" research in diffusion models is classical numerical ODE analysis — higher order, exponential integrators, and better step schedules — applied to an integrand that happens to be a neural network.

### Problem L2.6: Long-time energy behaviour on the harmonic oscillator

Integrate $\dot{q} = p$, $\dot{p} = -q$ (energy $E = \tfrac12(p^2 + q^2)$) with $h = 0.1$ for $1000$ steps using forward Euler, backward Euler, symplectic Euler, and RK4. Compute the energy amplification of each and explain the pattern.

**Solution.**

**Forward Euler.** The step map is $\begin{pmatrix} q\\p\end{pmatrix} \mapsto \begin{bmatrix} 1 & h \\ -h & 1\end{bmatrix}\begin{pmatrix} q\\p\end{pmatrix}$, and since $q_{n+1}^2 + p_{n+1}^2 = (1 + h^2)(q_n^2 + p_n^2)$ exactly,

$$
E_n = (1 + h^{2})^{n}E_0 = (1.01)^{1000}E_0 \approx 2.096 \times 10^{4}\,E_0 ,
$$

a $21000$-fold energy gain — the trajectory spirals outward without bound. Equivalently, $\det M_{\text{FE}} = 1 + h^2 \gt 1$: phase-space area is inflated every step.

**Backward Euler.** The inverse map, $E_n = (1+h^2)^{-n}E_0 \approx 4.77\times 10^{-5}E_0$: the oscillator is artificially damped to nothing. Note the irony — the A-stable method is *too* dissipative here, destroying the physics it was meant to preserve.

**Symplectic Euler** ($p_{n+1} = p_n - hq_n$, then $q_{n+1} = q_n + hp_{n+1}$). Its matrix $\begin{bmatrix} 1-h^2 & h \\ -h & 1\end{bmatrix}$ has determinant $(1-h^2) + h^2 = 1$ exactly, for every $h$. The energy is not conserved exactly, but the modified Hamiltonian $\tilde{H} = \tfrac12(p^2 + q^2 - hpq)$ **is**, so $E$ oscillates within a band of relative width $O(h) = 0.1$ and never drifts — the same after $10^{3}$ or $10^{9}$ steps.

**RK4.** $\vert R(ih)\vert^{2} = 1 - \tfrac{h^{6}}{72} + \tfrac{h^{8}}{576}$, so with $h = 0.1$ the per-step energy factor is $1 - 1.39\times 10^{-8}$, and after $1000$ steps

$$
\frac{E_{1000}}{E_0} = \left(1 - 1.39\times10^{-8}\right)^{1000} \approx 1 - 1.39\times 10^{-5} ,
$$

a loss of $0.0014\%$ — invisible over this span. But the loss is **secular**: it accumulates as $e^{-t h^{5}/72}$, so at $10^{7}$ steps ($t = 10^{6}$) the energy has fallen to $e^{-0.139} \approx 87\%$ of its initial value, a $13\%$ spurious dissipation.

| Method | Energy factor per step | After $1000$ steps | Long-time behaviour |
| :--- | :--- | :--- | :--- |
| Forward Euler | $1 + h^{2} = 1.01$ | $\times 2.10\times10^{4}$ | exponential growth |
| Backward Euler | $(1+h^{2})^{-1}$ | $\times 4.77\times10^{-5}$ | exponential decay |
| Symplectic Euler | $1$ (area exactly) | bounded $O(h)$ ripple | **no drift, ever** |
| Classical RK4 | $1 - h^{6}/72$ | $\times (1 - 1.39\times10^{-5})$ | slow secular decay |

$$
\boxed{\text{FE } \times 2.10\times 10^{4}, \quad \text{BE } \times 4.77\times10^{-5}, \quad \text{RK4 } \times (1 - 1.39\times10^{-5}), \quad \text{symplectic: bounded}}
$$

*Key takeaway:* Order controls the error over a *fixed* interval; structure preservation controls it over an *unbounded* one. A first-order symplectic method beats fourth-order RK4 for long Hamiltonian integrations — which is why molecular dynamics, celestial mechanics, and Hamiltonian Monte Carlo all use leapfrog.

## Level 3 — Challenge

### Problem L3.1: Global convergence of a general one-step method via the discrete Gronwall lemma

Prove that a one-step method $y_{n+1} = y_n + h\Phi(t_n, y_n, h)$ of order $p$, with $\Phi$ Lipschitz in $y$ with constant $L_\Phi$, converges with global order $p$, stating and proving the discrete Gronwall lemma along the way.

**Solution.**

**Lemma (discrete Gronwall).** If $E_{n+1} \le aE_n + C$ for $n \ge 0$ with $a \ge 1$, $C \ge 0$, then

$$
E_n \le a^{n}E_0 + C\,\frac{a^{n} - 1}{a - 1} \quad (a \gt 1) .
$$

*Proof of the lemma.* Induction. True at $n = 0$. Assuming it at $n$,

$$
E_{n+1} \le a\left(a^{n}E_0 + C\frac{a^{n}-1}{a-1}\right) + C = a^{n+1}E_0 + C\,\frac{a^{n+1} - a + a - 1}{a-1} = a^{n+1}E_0 + C\frac{a^{n+1}-1}{a-1} . \ \square
$$

**Step 1 — the defect of the exact solution.** By the definition of the local truncation error and order $p$, the exact solution satisfies

$$
y(t_{n+1}) = y(t_n) + h\,\Phi\bigl(t_n, y(t_n), h\bigr) + h\tau_{n+1}, \qquad \vert \tau_{n+1}\vert \le Ch^{p} .
$$

**Step 2 — the error recursion.** Subtract the scheme $y_{n+1} = y_n + h\Phi(t_n, y_n, h)$ and write $e_n = y_n - y(t_n)$:

$$
e_{n+1} = e_n + h\left[\Phi(t_n, y_n, h) - \Phi(t_n, y(t_n), h)\right] - h\tau_{n+1} .
$$

Taking norms and using the Lipschitz property of $\Phi$,

$$
\Vert e_{n+1}\Vert \le (1 + hL_\Phi)\Vert e_n\Vert + Ch^{p+1} .
$$

This is exactly the lemma's hypothesis with $a = 1 + hL_\Phi \gt 1$ and the constant $Ch^{p+1}$.

**Step 3 — apply the lemma and bound $a^{n}$.** With $E_0 = \Vert e_0 \Vert$ and $a - 1 = hL_\Phi$,

$$
\Vert e_n\Vert \le (1 + hL_\Phi)^{n}\Vert e_0\Vert + \frac{Ch^{p+1}}{hL_\Phi}\left[(1+hL_\Phi)^{n} - 1\right] .
$$

Since $1 + x \le e^{x}$ for all real $x$, $(1 + hL_\Phi)^{n} \le e^{nhL_\Phi} = e^{L_\Phi(t_n - t_0)} \le e^{L_\Phi(T-t_0)}$. Hence, for exact initial data $e_0 = 0$,

$$
\max_{0 \le n \le N}\Vert e_n \Vert \le \frac{Ch^{p}}{L_\Phi}\left(e^{L_\Phi(T - t_0)} - 1\right) = O(h^{p}) . \qquad \blacksquare
$$

**Three remarks.** (i) The exponent $p$ — not $p+1$ — is where the "one lost order" comes from: the $h^{p+1}$ local error is divided by $h$ when the geometric sum is evaluated, because there are $O(1/h)$ terms. (ii) The factor $e^{L_\Phi(T-t_0)}$ is why long integrations of expanding systems are hopeless: with $L_\Phi(T-t_0) = 30$, the constant is $10^{13}$ and no attainable $h$ rescues the bound. (iii) With inexact initial data ($e_0 \neq 0$) the first term shows that the *initial* error is amplified by the same exponential — the numerical method inherits the conditioning of the ODE itself, and cannot do better than it.

$$
\boxed{\Vert e_n \Vert \le \Vert e_0\Vert e^{L_\Phi(t_n-t_0)} + \frac{Ch^{p}}{L_\Phi}\left(e^{L_\Phi(t_n - t_0)} - 1\right) = O(h^{p})}
$$

*Key takeaway:* Consistency of order $p$ plus a Lipschitz increment function gives convergence of order $p$ — the one-step half of Dahlquist's equivalence theorem, and the proof is nothing but a geometric series plus $1 + x \le e^{x}$.

### Problem L3.2: The trapezoidal rule — order 2, A-stable, and not L-stable

Prove that the trapezoidal rule has order 2, that its region of absolute stability is *exactly* the closed left half-plane, and that it is not L-stable. Explain the practical consequence for very stiff modes.

**Solution.**

**Order.** With $y_{n+1} = y_n + \tfrac{h}{2}[f(t_n,y_n) + f(t_{n+1},y_{n+1})]$, insert the exact solution and expand about $t_n$. Writing $g(t) = f(t, y(t)) = y'(t)$, the method's right-hand side is the trapezoidal quadrature of $\int_{t_n}^{t_{n+1}} g$, whose error is the classical

$$
\int_{t_n}^{t_{n+1}} g(s)\,ds - \frac{h}{2}\left[g(t_n) + g(t_{n+1})\right] = -\frac{h^{3}}{12}g''(\xi) = -\frac{h^{3}}{12}y'''(\xi) .
$$

Hence the one-step defect is $O(h^{3})$ and $\tau_{n+1} = -\tfrac{h^{2}}{12}y'''(\xi) = O(h^{2})$: **order 2**, with error constant $-1/12$ — the smallest possible among A-stable linear multistep methods (second Dahlquist barrier).

**Stability function.** On $y' = \lambda y$ with $z = h\lambda$:

$$
y_{n+1} = y_n + \frac{z}{2}(y_n + y_{n+1}) \implies \left(1 - \frac{z}{2}\right)y_{n+1} = \left(1 + \frac{z}{2}\right)y_n \implies R(z) = \frac{1 + z/2}{1 - z/2} .
$$

(This is the $(1,1)$ Padé approximant of $e^{z}$, matching its Taylor series through $z^2$ — confirming order 2 independently.)

**A-stability, exactly.** Write $w = z/2$. Then

$$
\vert R(z)\vert \le 1 \iff \vert 1 + w\vert \le \vert 1 - w\vert \iff \vert 1+w\vert^{2} \le \vert 1-w\vert^{2} \iff 1 + 2\operatorname{Re}w + \vert w\vert^{2} \le 1 - 2\operatorname{Re}w + \vert w\vert^{2},
$$

i.e. $4\operatorname{Re}w \le 0 \iff \operatorname{Re}z \le 0$. So the stability region is **exactly** the closed left half-plane $\overline{\mathbb{C}^{-}}$ — no more, no less. Geometrically: $\vert 1+w\vert \le \vert 1-w\vert$ says $w$ is at least as close to $-1$ as to $+1$, which is precisely the left half-plane. In particular, on the imaginary axis $\vert R\vert = 1$ exactly, so pure oscillations are neither damped nor amplified — the trapezoidal rule is the natural choice for conservative wave problems (where it is called Crank–Nicolson).

**Not L-stable.**

$$
\lim_{\operatorname{Re}z \to -\infty}R(z) = \lim \frac{1 + z/2}{1 - z/2} = -1 \neq 0 .
$$

**Practical consequence.** For a very stiff mode with $h\lambda = -10^{6}$, the exact solution decays by $e^{-10^{6}} \approx 0$ in one step, but the trapezoidal rule multiplies it by $R \approx -1$: the mode is *retained at full amplitude with an alternating sign*. The result is the familiar Crank–Nicolson **ringing** — spurious high-frequency oscillations in the stiff components that decay only algebraically. Backward Euler, with $R \to 0$, annihilates them instead. Production stiff solvers therefore prefer L-stable schemes (BDF2, Radau IIA, TR-BDF2 — which alternates a trapezoidal and a BDF2 stage to get both order 2 and L-stability).

$$
\boxed{R(z) = \frac{1+z/2}{1-z/2}, \quad \mathcal{S} = \{\operatorname{Re}z \le 0\} \text{ exactly}, \quad \tau = -\tfrac{h^2}{12}y''', \quad R(-\infty) = -1 \text{ (not L-stable)}}
$$

*Key takeaway:* A-stability says stiff modes will not blow up; L-stability says they will be *removed*. On genuinely stiff problems that difference is the difference between a clean solution and a ringing one.

### Problem L3.3: High order is worthless without zero-stability

The two-step method $y_{n+2} + 4y_{n+1} - 5y_n = h\left(4f_{n+1} + 2f_n\right)$ has order 3. Verify the order, test the root condition, and demonstrate the failure on $y' = 0$.

**Solution.**

**Order verification.** Insert the exact solution and expand about $t_n$, writing $y = y(t_n)$, $y' = y'(t_n)$, etc. Using $y(t_n + jh) = y + jhy' + \tfrac{(jh)^2}{2}y'' + \tfrac{(jh)^3}{6}y''' + O(h^4)$ and $f_{n+j} = y'(t_n + jh)$:

$$
\begin{aligned}
y_{n+2} + 4y_{n+1} - 5y_n &= (1 + 4 - 5)y + (2 + 4)hy' + \left(2 + 2\right)h^2y'' + \left(\tfrac{8}{6} + \tfrac{4}{6}\right)h^3y''' + O(h^4) \\
&= 6hy' + 4h^{2}y'' + 2h^{3}y''' + O(h^{4}), \\
h(4f_{n+1} + 2f_n) &= h\left[4\left(y' + hy'' + \tfrac{h^2}{2}y'''\right) + 2y'\right] + O(h^{4}) = 6hy' + 4h^{2}y'' + 2h^{3}y''' + O(h^{4}).
\end{aligned}
$$

At order $h^{4}$ the coefficients are $\tfrac{5}{6}$ on the left and $\tfrac{4}{6}$ on the right, so the defect is $\tfrac{h^{4}}{6}y^{(4)} + O(h^{5})$ and the method has **order 3**. That is already the warning sign: the first Dahlquist barrier caps a *zero-stable explicit* $k$-step method at order $k$, so order 3 from a 2-step explicit method can only mean zero-stability has been sacrificed.

**Root condition.** The first characteristic polynomial is

$$
\rho(\zeta) = \zeta^{2} + 4\zeta - 5 = (\zeta - 1)(\zeta + 5) .
$$

The root $\zeta_1 = 1$ is the *principal* root, required by consistency. The parasitic root $\zeta_2 = -5$ has $\vert\zeta_2\vert = 5 \gt 1$: **the root condition fails and the method is not zero-stable.**

**Demonstration on $y' = 0$, $y(0) = 1$.** Here $f \equiv 0$ and the recursion is purely homogeneous: $y_{n+2} = -4y_{n+1} + 5y_n$, whose general solution is $y_n = A\cdot 1^{n} + B(-5)^{n}$. With *exact* starting values $y_0 = y_1 = 1$ we get $A = 1$, $B = 0$ and the method reproduces $y_n \equiv 1$ perfectly. But suppose $y_1 = 1 + \delta$ with $\delta = 10^{-16}$ (one rounding unit). Matching, $A + B = 1$ and $A - 5B = 1 + \delta$ give $B = -\delta/6$, so

$$
y_n = 1 - \frac{\delta}{6}(-5)^{n} .
$$

At $n = 20$: $\vert y_{20} - 1\vert = \tfrac{10^{-16}}{6}\cdot 5^{20} \approx \tfrac{10^{-16}}{6}\cdot 9.54\times 10^{13} \approx 1.6\times 10^{-3}$. At $n = 40$, with $5^{40} \approx 9.09\times 10^{27}$, the perturbation is $\approx 1.5\times 10^{11}$. **The answer is destroyed** — and crucially, *refining $h$ does not help*, because the growth $(-5)^{n}$ depends only on the number of steps, and smaller $h$ means *more* steps.

$$
\boxed{\text{order } 3, \text{ but } \rho(\zeta) = (\zeta-1)(\zeta+5) \text{ has } \vert\zeta\vert = 5 \gt 1: \text{ divergent, } y_n = 1 - \tfrac{\delta}{6}(-5)^{n}}
$$

*Key takeaway:* This is the concrete content of Dahlquist's equivalence theorem: consistency alone does not imply convergence. Zero-stability is the missing hypothesis, and violating it produces a method that is *more* accurate per step and *catastrophically* wrong overall.

### Problem L3.4: Symplectic Euler conserves a modified Hamiltonian exactly

For the harmonic oscillator, prove that symplectic Euler is area-preserving and that the modified Hamiltonian $\tilde{H}(q,p) = \tfrac12\left(p^{2} + q^{2} - hpq\right)$ is conserved *exactly*. Deduce the bounded-energy property.

**Solution.**

**The map.** Symplectic Euler for $H = \tfrac12(p^2 + q^2)$ updates the momentum with the *old* position and the position with the *new* momentum:

$$
p_{n+1} = p_n - hq_n, \qquad q_{n+1} = q_n + hp_{n+1} = q_n + h(p_n - hq_n) = (1-h^{2})q_n + hp_n .
$$

In matrix form on $(q, p)$:

$$
M = \begin{bmatrix} 1 - h^{2} & h \\ -h & 1\end{bmatrix} .
$$

**Step 1 — area preservation.** $\det M = (1-h^{2})\cdot 1 - h\cdot(-h) = 1 - h^{2} + h^{2} = 1$, exactly, for every $h$. In one degree of freedom, $\det M = 1$ *is* the symplectic condition $M^{\top}JM = J$ with $J = \begin{bmatrix} 0 & 1 \\ -1 & 0\end{bmatrix}$ — indeed $M^{\top}JM = (\det M)J$ for any $2\times2$ matrix. Contrast forward Euler ($\det = 1 + h^{2}$, expanding) and backward Euler ($\det = (1+h^2)^{-1}$, contracting).

**Step 2 — the conserved quadratic form.** Let $\tilde{H} = \tfrac12(q^{2} + p^{2} - hqp)$ and evaluate it after one step. With $q' = (1-h^2)q + hp$ and $p' = -hq + p$:

$$
\begin{aligned}
q'^{2} &= (1-h^{2})^{2}q^{2} + 2h(1-h^{2})qp + h^{2}p^{2}, \\
p'^{2} &= h^{2}q^{2} - 2hqp + p^{2}, \\
-hq'p' &= h^{2}(1-h^{2})q^{2} - h(1-h^{2})qp + h^{3}qp - h^{2}p^{2} .
\end{aligned}
$$

Collect coefficients:

- $q^{2}$: $(1 - 2h^{2} + h^{4}) + h^{2} + h^{2} - h^{4} = 1$.
- $p^{2}$: $h^{2} + 1 - h^{2} = 1$.
- $qp$: $2h(1-h^{2}) - 2h - h(1-h^{2}) + h^{3} = h(1-h^{2}) - 2h + h^{3} = h - h^{3} - 2h + h^{3} = -h$.

So $q'^{2} + p'^{2} - hq'p' = q^{2} + p^{2} - hqp$: the form $\tilde{H}$ is **exactly invariant**. $\blacksquare$

**Step 3 — bounded energy.** $\tilde{H} = H - \tfrac{h}{2}qp$, so $\vert \tilde{H} - H \vert = \tfrac{h}{2}\vert qp\vert \le \tfrac{h}{2}\cdot\tfrac{q^2+p^2}{2} = \tfrac{h}{2}H$. Since $\tilde{H}$ is constant along the numerical trajectory,

$$
\frac{\tilde{H}_0}{1 + h/2} \le H_n \le \frac{\tilde{H}_0}{1 - h/2} \quad \text{for } h \lt 2 ,
$$

so the true energy stays inside a band of relative width $O(h)$ **for all $n$** — no drift, ever. (Positive definiteness of $\tilde{H}$ for $h \lt 2$ also confirms the numerical orbits are closed ellipses rather than spirals, and that the method is stable.) For $h = 0.1$ the band is about $\pm 5\%$: the energy wobbles but never escapes, in sharp contrast to forward Euler's $(1+h^{2})^{n}$ growth or RK4's slow secular decay.

**The general statement.** Backward error analysis shows that any symplectic method of order $p$ applied to a smooth Hamiltonian system is the exact flow of a modified Hamiltonian $\tilde{H} = H + h^{p}H_{p} + h^{p+1}H_{p+1} + \cdots$, with the truncated series conserved to exponentially small accuracy $O(e^{-c/h})$ over times $O(e^{c/h})$. The quadratic $\tilde H$ above is the exactly summable instance of that series for a linear system.

$$
\boxed{\det M = 1, \qquad \tilde{H} = \tfrac12(q^{2}+p^{2}-hqp) \text{ conserved exactly}, \qquad \vert H_n - H_0\vert = O(h)H_0 \ \forall n}
$$

*Key takeaway:* A symplectic method does not conserve the energy — it conserves a *nearby* energy, exactly and forever. That is a far stronger long-time guarantee than any order estimate, and it is why order 2 symplectic beats order 4 non-symplectic over long horizons.